In [ ]:
import * as tslab from "tslab";

# The Bridge and the Torch

In the darkness of the night, a group of four individuals encounters a river. A slender bridge stretches before them, capable of accommodating just two people simultaneously. Equipped with a single torch, they must rely on its flickering light to navigate the bridge. Each person possesses a distinct crossing time: Ariela takes 1 minute, Brian takes 2 minutes, Charly takes 5 minutes, and Dumpy takes 8 minutes. It is crucial to note that when two people cross together, they must synchronize their steps with the slower individual's pace. Given the torch's limited lifespan of 15 minutes, the pressing question arises: can all four individuals successfully traverse the bridge?

In [ ]:
import { init, Bool, Arith } from 'z3-solver';
const { Context } = await init();
const Z3 = Context("main");

We need the following variables to encode the problem:
* `A` equals `1` if Ariela is on the left shore,
* `B` equals `1` if Brian  is on the left shore,
* `C` equals `1` if Charly is on the left shore,
* `D` equals `1` if Dumpy  is on the left shore,
* `T` equals the time that has passed.

`start` returns a set of constraints that specify that everybody is on the left side of the river.
Furthermore, no time has passed.
$$ \texttt{A} = 1 \wedge \texttt{B} = 1 \wedge \texttt{C} = 1 \wedge \texttt{D} = 1 \wedge \texttt{T} = 0 $$

In [ ]:
function start(A: Arith, B: Arith, C: Arith, D: Arith, T: Arith): Bool[] {
    return "your code here";
}

`goal` returns a set of constraints that specify that everybody is on the right side of the river.
The time must be less or equal than $15$:
$$ \texttt{A} = 0 \wedge \texttt{B} = 0 \wedge \texttt{C} = 0 \wedge \texttt{D} = 0 \wedge \texttt{T} \leq 15 $$

In [ ]:
function goal(A: Arith, B: Arith, C: Arith, D: Arith, T: Arith): Bool[] {
    return "your code here";
}

`torchOK` descrives one crossing of the bridge from left to right.
* `A` is the number of Arielas on the left side before the crossing, 
   while `Ax` is the number of Arielas on the left side after the crossing.
* Dito for `B`, `Bx`, $\cdots$, `D`, `Dx`.
* `T` is the time before the crossing and `Tx` is the time after the crossing.

The constraints ensure that
 * People go from left to right,
 * at least one person crosses the bridge,
 * at most two persons cross the bridge,
 * the variable `Tx` is incremented by the right amount.

In [ ]:
function torchOK(A : Arith, B : Arith, C : Arith, D : Arith, T : Arith, 
                 Ax: Arith, Bx: Arith, Cx: Arith, Dx: Arith, Tx: Arith): Bool[] {
    "your code here"
}

`transition` returns a set of constraints that describe the crossings of the bridge.
The variable `i` specifies the number of the crossing.  The first crossing has `i == 0`.

In [ ]:
function transition(A : Arith, B : Arith, C : Arith, D : Arith, T : Arith, 
                    Ax: Arith, Bx: Arith, Cx: Arith, Dx: Arith, Tx: Arith, i: number): Bool[] {
    if (i % 2 === 0) {
        "your code here";
    } else {
        "your code here";
    }
}

`bridge_CSP` tries to solve the problem using `n` crossings of the bridge.
If this is possible, a solution is returned that is a dictionary mapping the variables 
to their values.  Otherwise, `null` is returned.

In [ ]:
import { RecursiveMap as Map } from "recursive-set";

In [ ]:
async function bridge_CSP(n: number): Promise<Map<string, number>> {
    const solver = new Z3.Solver();
    const As: Arith[] = [], Bs: Arith[] = [], Cs: Arith[] = [], Ds: Arith[] = [], Ts: Arith[] = [];
    
    for (let i = 0; i <= n; i++) {
        As.push(Z3.Int.const(`A${i}`));
        Bs.push(Z3.Int.const(`B${i}`));
        Cs.push(Z3.Int.const(`C${i}`));
        Ds.push(Z3.Int.const(`D${i}`));
        Ts.push(Z3.Int.const(`T${i}`));
    }
    solver.add(...start(As[0], Bs[0], Cs[0], Ds[0], Ts[0]));
    solver.add(...goal (As[n], Bs[n], Cs[n], Ds[n], Ts[n]));
    for (let i = 0; i < n; i++) {
        solver.add(...transition(As[i  ], Bs[i  ], Cs[i  ], Ds[i  ], Ts[i  ], 
                                 As[i+1], Bs[i+1], Cs[i+1], Ds[i+1], Ts[i+1], i));
        // Bounds
        solver.add(As[i].ge(0), As[i].le(1));
        solver.add(Bs[i].ge(0), Bs[i].le(1));
        solver.add(Cs[i].ge(0), Cs[i].le(1));
        solver.add(Ds[i].ge(0), Ds[i].le(1));
        solver.add(Ts[i].ge(0));
    }
    // Bounds for the final step
    solver.add(As[n].ge(0), As[n].le(1));
    solver.add(Bs[n].ge(0), Bs[n].le(1));
    solver.add(Cs[n].ge(0), Cs[n].le(1));
    solver.add(Ds[n].ge(0), Ds[n].le(1));
    solver.add(Ts[n].ge(0));
    
    const result = await solver.check();
    if (result === 'sat') {
        const model = solver.model();
        const solution = new Map<string, number>();
        for (let i = 0; i <= n; i++) {
            solution.set(`A${i}`, parseInt(model.eval(As[i]).toString(), 10));
            solution.set(`B${i}`, parseInt(model.eval(Bs[i]).toString(), 10));
            solution.set(`C${i}`, parseInt(model.eval(Cs[i]).toString(), 10));
            solution.set(`D${i}`, parseInt(model.eval(Ds[i]).toString(), 10));
            solution.set(`T${i}`, parseInt(model.eval(Ts[i]).toString(), 10));
        }
        return solution;
    }
}

In [ ]:
async function findSolution(): Promise<{n: number, solution: Map<string, number>}> {
    let n = 1;
    while (true) {
        console.log(n);
        const solution = await bridge_CSP(n);
        if (solution) {
            return { n, solution };
        }
        n += 2;
    }
}

In [ ]:
console.time("Bridge Solving");
const result = await findSolution();
console.timeEnd("Bridge Solving");
result

In [ ]:
function show_solution(sol: Map<string, number>, n: number) {
    for (let i = 0; i <= n; i++) {
        const A = sol.get(`A${i}`)!;
        const B = sol.get(`B${i}`)!;
        const C = sol.get(`C${i}`)!;
        const D = sol.get(`D${i}`)!;
        const T = sol.get(`T${i}`)!;
        
        const left = '🏃‍♀️'.repeat(A) + '🏃🏽‍♂️'.repeat(B) + '🚶🏽‍♂️'.repeat(C) + '👨‍🦽'.repeat(D);
        const right = '🏃‍♀️'.repeat(1-A) + '🏃🏽‍♂️'.repeat(1-B) + '🚶🏽‍♂️'.repeat(1-C) + '👨‍🦽'.repeat(1-D);
        console.log(left.padEnd(20) + ' '.repeat(22) + right);
        console.log(`🕰️${T}`);
        
        if (i % 2 === 0 && i < n) {
            const PS = sol.get(`A${i}`)! - sol.get(`A${i+1}`)!;
            const BS = sol.get(`B${i}`)! - sol.get(`B${i+1}`)!;
            const CS = sol.get(`C${i}`)! - sol.get(`C${i+1}`)!;
            const FS = sol.get(`D${i}`)! - sol.get(`D${i+1}`)!;
            const cross = '🏃‍♀️'.repeat(PS) + '🏃🏽‍♂️'.repeat(BS) + '🚶🏽‍♂️'.repeat(CS) + '👨‍🦽'.repeat(FS);
            console.log(' '.repeat(24) + '>> ' + cross + ' >>');
        } else if (i + 1 <= n) {
            const PS = sol.get(`A${i+1}`)! - sol.get(`A${i}`)!;
            const BS = sol.get(`B${i+1}`)! - sol.get(`B${i}`)!;
            const CS = sol.get(`C${i+1}`)! - sol.get(`C${i}`)!;
            const FS = sol.get(`D${i+1}`)! - sol.get(`D${i}`)!;
            const cross = '🏃‍♀️'.repeat(PS) + '🏃🏽‍♂️'.repeat(BS) + '🚶🏽‍♂️'.repeat(CS) + '👨‍🦽'.repeat(FS);
            console.log(' '.repeat(24) + '<< ' + cross + ' <<');
        }
    }
}

In [ ]:
show_solution(result.solution, result.n);

In [ ]:
function show_animated_solution(sol: Map<string, number>, n: number) {
    if (!sol) return;
    const states = [];
    for (let i = 0; i <= n; i++) {
        states.push({ 
            A: sol.get(`A${i}`), B: sol.get(`B${i}`), 
            C: sol.get(`C${i}`), D: sol.get(`D${i}`),
            T: sol.get(`T${i}`), 
            torchLeft: i % 2 === 0 
        });
    }
    const uid = Math.random().toString(36).substring(2, 9);
    const htmlContent = `
    <div style="font-family: sans-serif; max-width: 600px; margin: 0 auto; text-align: center;">
        <div style="margin-bottom: 15px;">
            <button id="btn-reset-${uid}" style="padding: 8px 16px; cursor: pointer;">Reset</button>
            <button id="btn-prev-${uid}" style="padding: 8px 16px; cursor: pointer;">Previous Step</button>
            <button id="btn-next-${uid}" style="padding: 8px 16px; cursor: pointer; font-weight: bold;">Next Step</button>
            <span id="label-step-${uid}" style="margin-left: 15px; font-weight: bold; font-size: 16px;">Step 0 of ${n}</span>
        </div>
        <div style="margin-bottom: 10px; font-size: 18px; font-weight: bold; color: #d32f2f;">Time Elapsed: <span id="timer-${uid}">0</span> mins</div>
        <svg width="600" height="300" style="background-color: #0b1120; border: 1px solid #333; border-radius: 8px;">
            <rect x="0" y="0" width="150" height="300" fill="#1e3a29" />
            <rect x="450" y="0" width="150" height="300" fill="#1e3a29" />
            <rect x="150" y="130" width="300" height="40" fill="#5c4033" stroke="#3b261e" stroke-width="4" />
            
            <text x="75" y="30" font-size="14" font-weight="bold" fill="#a5d6a7" text-anchor="middle">Left Shore</text>
            <text x="525" y="30" font-size="14" font-weight="bold" fill="#a5d6a7" text-anchor="middle">Right Shore</text>
            
            <text id="west-a-${uid}" x="75" y="100" font-size="24" text-anchor="middle"></text>
            <text id="west-b-${uid}" x="75" y="140" font-size="24" text-anchor="middle"></text>
            <text id="west-c-${uid}" x="75" y="180" font-size="24" text-anchor="middle"></text>
            <text id="west-d-${uid}" x="75" y="220" font-size="24" text-anchor="middle"></text>

            <text id="east-a-${uid}" x="525" y="100" font-size="24" text-anchor="middle"></text>
            <text id="east-b-${uid}" x="525" y="140" font-size="24" text-anchor="middle"></text>
            <text id="east-c-${uid}" x="525" y="180" font-size="24" text-anchor="middle"></text>
            <text id="east-d-${uid}" x="525" y="220" font-size="24" text-anchor="middle"></text>

            <g id="group-${uid}" style="transition: transform 1.2s ease-in-out;">
                <text x="200" y="110" font-size="28" text-anchor="middle">🔦</text>
                <text id="bridge-p-${uid}" x="200" y="160" font-size="24" text-anchor="middle"></text>
            </g>
        </svg>
    </div>
    <script>
    (function() {
        const states = ${JSON.stringify(states)};
        let currentStep = 0;
        let isAnimating = false;
        
        const group = document.getElementById('group-${uid}');
        const wA = document.getElementById('west-a-${uid}');
        const wB = document.getElementById('west-b-${uid}');
        const wC = document.getElementById('west-c-${uid}');
        const wD = document.getElementById('west-d-${uid}');
        const eA = document.getElementById('east-a-${uid}');
        const eB = document.getElementById('east-b-${uid}');
        const eC = document.getElementById('east-c-${uid}');
        const eD = document.getElementById('east-d-${uid}');
        const bridgeP = document.getElementById('bridge-p-${uid}');
        const stepLabel = document.getElementById('label-step-${uid}');
        const timerLabel = document.getElementById('timer-${uid}');
        
        function renderStaticState(step) {
            const s = states[step];
            wA.textContent = '🏃‍♀️'.repeat(s.A);
            wB.textContent = '🏃🏽‍♂️'.repeat(s.B);
            wC.textContent = '🚶🏽‍♂️'.repeat(s.C);
            wD.textContent = '👨‍🦽'.repeat(s.D);
            eA.textContent = '🏃‍♀️'.repeat(1 - s.A);
            eB.textContent = '🏃🏽‍♂️'.repeat(1 - s.B);
            eC.textContent = '🚶🏽‍♂️'.repeat(1 - s.C);
            eD.textContent = '👨‍🦽'.repeat(1 - s.D);
            bridgeP.textContent = '';
            
            group.style.transform = s.torchLeft ? 'translateX(0px)' : 'translateX(200px)';
            stepLabel.textContent = 'Step ' + step + ' of ${n}';
            timerLabel.textContent = s.T;
        }

        async function animateTransition(fromStep, toStep) {
            if (isAnimating) return;
            isAnimating = true;
            
            const s0 = states[fromStep];
            const s1 = states[toStep];
            
            const aPass = Math.abs(s0.A - s1.A);
            const bPass = Math.abs(s0.B - s1.B);
            const cPass = Math.abs(s0.C - s1.C);
            const dPass = Math.abs(s0.D - s1.D);
            
            if (s0.torchLeft) { 
                wA.textContent = '🏃‍♀️'.repeat(s0.A - aPass);
                wB.textContent = '🏃🏽‍♂️'.repeat(s0.B - bPass);
                wC.textContent = '🚶🏽‍♂️'.repeat(s0.C - cPass);
                wD.textContent = '👨‍🦽'.repeat(s0.D - dPass);
            } else { 
                eA.textContent = '🏃‍♀️'.repeat((1 - s0.A) - aPass);
                eB.textContent = '🏃🏽‍♂️'.repeat((1 - s0.B) - bPass);
                eC.textContent = '🚶🏽‍♂️'.repeat((1 - s0.C) - cPass);
                eD.textContent = '👨‍🦽'.repeat((1 - s0.D) - dPass);
            }
            bridgeP.textContent = '🏃‍♀️'.repeat(aPass) + '🏃🏽‍♂️'.repeat(bPass) + '🚶🏽‍♂️'.repeat(cPass) + '👨‍🦽'.repeat(dPass);
            
            group.style.transform = s1.torchLeft ? 'translateX(0px)' : 'translateX(200px)';
            
            await new Promise(r => setTimeout(r, 1200));
            
            currentStep = toStep;
            renderStaticState(currentStep);
            isAnimating = false;
        }

        document.getElementById('btn-next-${uid}').addEventListener('click', () => {
            if (currentStep < states.length - 1) animateTransition(currentStep, currentStep + 1);
        });
        document.getElementById('btn-prev-${uid}').addEventListener('click', () => {
            if (currentStep > 0) animateTransition(currentStep, currentStep - 1);
        });
        document.getElementById('btn-reset-${uid}').addEventListener('click', () => {
            if (!isAnimating) {
                currentStep = 0;
                renderStaticState(0);
            }
        });

        renderStaticState(0);
    })();
    </script>
    `;
    tslab.display.html(htmlContent);
}

In [ ]:
show_animated_solution(result.solution, result.n);